# Match Tactical Stats + Scoreboard/Clock OCR (per image, no tracking)

Builds on the detection metadata + team classification + homography work already done. For each
qualifying image, computes:

- **Team shape**: centroid, width, depth, convex hull area, bounding box, aspect ratio
- **Spacing**: avg nearest-teammate distance, avg distance to team centroid, inter-team centroid
  distance, defensive line height (deepest outfield player, per team)
- **Zone occupation**: player counts by pitch thirds (left/middle/right along each axis) —
  reported as absolute pitch zones, NOT "attacking third", since which goal each team is
  attacking isn't knowable from a single image without extra info
- **Ball-relative**: nearest player + team, player counts within a radius of the ball, ball
  distance to each goal
- **Scoreboard / clock OCR**: reads on-screen score and match-clock text, IF you've set the pixel
  region they appear in (see CONFIG) — skipped (not guessed) otherwise

## What this deliberately does NOT include
Goals, shots, and possession ratio are not computed here — they're cumulative/event statistics
that need either ordered video frames (goals, via scoreboard-diffing) or full tracking (shots,
possession), both ruled out per your discrete-image requirement. See the conversation notes if
you want to revisit that later.

## Setup: Tesseract OCR engine (separate from the `pytesseract` Python package)
`pip install pytesseract` only installs a Python wrapper — it calls out to the actual Tesseract
OCR engine, which must be installed separately on your PC:
- **Windows**: install from https://github.com/UB-Mannheim/tesseract/wiki, then set
  `pytesseract.pytesseract.tesseract_cmd` to its install path (see CONFIG below).
- **Mac**: `brew install tesseract`
- **Linux**: `sudo apt install tesseract-ocr`

If this isn't installed, OCR calls will fail — the score/clock fields will just come back as
`None` with a printed warning, not crash the whole pipeline.


In [1]:
%pip install -q opencv-python-headless pytesseract --upgrade


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Users\nishi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.cluster import KMeans

try:
    import pytesseract
    OCR_AVAILABLE = True
except ImportError:
    OCR_AVAILABLE = False
    print("pytesseract not installed — OCR will be skipped entirely.")


In [3]:
# ============================================================
# CONFIG
# ============================================================
CONFIG = {
    "metadata_paths": {
        "train": "./metadata/train_metadata.jsonl",
        "test": "./metadata/test_metadata.jsonl",
    },
    "stats_output_dir": "./tactical_stats",

    "min_person_count": 5,
    "min_players_for_team_split": 4,

    "homography_mode": "auto",   # "auto" or "manual" — same as the bird's-eye-view notebook
    "manual_homography_src": None,   # [[top-left],[top-right],[bottom-right],[bottom-left]] pixel coords

    "ball_radius_m": 10,   # radius (meters, on the pitch) used for "players near ball" counts

    # --- OCR regions, in PIXEL coordinates [x1, y1, x2, y2] of a representative image ---
    # Leave as None to skip that OCR field entirely (no guessing). Find these coordinates by
    # opening one frame in any image viewer/editor and reading off where the scoreboard/clock
    # graphic sits.
    "score_roi": None,     # e.g. [820, 20, 980, 55]
    "clock_roi": None,     # e.g. [600, 20, 700, 55]

    # If your Tesseract install isn't on PATH (common on Windows), set its full path here.
    "tesseract_cmd_path": None,   # e.g. r"C:\\Program Files\\Tesseract-OCR\\tesseract.exe"
}

PERSON_CLASSES = {"player", "goalkeeper", "referee"}
PITCH_W, PITCH_H = 105, 68

if CONFIG["homography_mode"] == "manual" and not CONFIG["manual_homography_src"]:
    raise RuntimeError("homography_mode is 'manual' but manual_homography_src is not set.")

if OCR_AVAILABLE and CONFIG["tesseract_cmd_path"]:
    pytesseract.pytesseract.tesseract_cmd = CONFIG["tesseract_cmd_path"]

os.makedirs(CONFIG["stats_output_dir"], exist_ok=True)


In [4]:
# ============================================================
# Load metadata (unchanged from earlier notebooks).
# ============================================================
def load_metadata(path):
    records = []
    if not os.path.exists(path):
        print(f"WARNING: {path} not found — skipping.")
        return records
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            if "error" in record:
                continue
            records.append(record)
    return records


In [5]:
# ============================================================
# Team + goalkeeper assignment (unchanged from the bird's-eye-view notebook).
# ============================================================
def get_jersey_color(image_hsv, bbox_px):
    w, h = image_hsv.size
    x1, y1, x2, y2 = [int(round(v)) for v in bbox_px]
    box_h = y2 - y1
    ty1 = max(0, y1 + int(0.15 * box_h))
    ty2 = max(ty1 + 1, y1 + int(0.55 * box_h))
    x1, x2 = max(0, x1), min(w, max(x1 + 1, x2))
    ty2 = min(h, ty2)
    crop = image_hsv.crop((x1, ty1, x2, ty2))
    arr = np.array(crop).reshape(-1, 3).astype(np.float32)
    if arr.size == 0:
        return np.array([0.0, 0.0, 0.0])
    return arr.mean(axis=0)


def assign_teams(image_pil, detections, min_players_for_split):
    image_hsv = image_pil.convert("HSV")
    for d in detections:
        d["team"] = None

    player_dets = [d for d in detections if d["class"] == "player"]
    goalkeeper_dets = [d for d in detections if d["class"] == "goalkeeper"]

    if len(player_dets) >= min_players_for_split:
        features = np.array([get_jersey_color(image_hsv, d["bbox_px"]) for d in player_dets])
        kmeans = KMeans(n_clusters=2, n_init=10, random_state=0).fit(features)
        hue_by_cluster = {c: features[kmeans.labels_ == c][:, 0].mean() for c in range(2)}
        ordered = sorted(hue_by_cluster, key=hue_by_cluster.get)
        label_map = {ordered[0]: "red", ordered[1]: "blue"}
        for d, cluster in zip(player_dets, kmeans.labels_):
            d["team"] = label_map[cluster]
    else:
        for d in player_dets:
            d["team"] = "unknown"

    teamed_players = [d for d in player_dets if d["team"] in ("red", "blue")]
    for gk in goalkeeper_dets:
        if not teamed_players:
            gk["team"] = "unknown"
            continue
        gx, gy = gk["ground_point_px"]
        nearest = min(teamed_players, key=lambda p: (p["ground_point_px"][0] - gx) ** 2
                                                     + (p["ground_point_px"][1] - gy) ** 2)
        gk["team"] = nearest["team"]

    return detections


In [6]:
# ============================================================
# Homography (unchanged from the bird's-eye-view notebook).
# ============================================================
def segment_pitch_mask(image_bgr):
    h, w = image_bgr.shape[:2]
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    cy0, cy1 = int(h * 0.3), int(h * 0.85)
    cx0, cx1 = int(w * 0.2), int(w * 0.8)
    sample = hsv[cy0:cy1, cx0:cx1].reshape(-1, 3)
    median_hue = np.median(sample[:, 0])
    lower = np.array([max(0, median_hue - 25), 30, 30], dtype=np.uint8)
    upper = np.array([min(179, median_hue + 25), 255, 255], dtype=np.uint8)
    mask = cv2.inRange(hsv, lower, upper)
    kernel = np.ones((15, 15), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    largest = max(contours, key=cv2.contourArea)
    if cv2.contourArea(largest) < 0.05 * h * w:
        return None
    return largest


def order_points(pts):
    pts = np.array(pts, dtype=np.float32)
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1).flatten()
    top_left = pts[np.argmin(s)]
    bottom_right = pts[np.argmax(s)]
    top_right = pts[np.argmin(diff)]
    bottom_left = pts[np.argmax(diff)]
    return np.array([top_left, top_right, bottom_right, bottom_left], dtype=np.float32)


def detect_pitch_quadrilateral(image_bgr):
    contour = segment_pitch_mask(image_bgr)
    if contour is None:
        return None
    peri = cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, 0.02 * peri, True)
    if len(approx) == 4:
        pts = approx.reshape(4, 2)
    else:
        rect = cv2.minAreaRect(contour)
        pts = cv2.boxPoints(rect)
    return order_points(pts)


CANONICAL_DST = np.array([[0, PITCH_H], [PITCH_W, PITCH_H], [PITCH_W, 0], [0, 0]], dtype=np.float32)

def compute_homography(image_bgr):
    quad = detect_pitch_quadrilateral(image_bgr)
    if quad is None:
        return None
    H, _ = cv2.findHomography(quad, CANONICAL_DST)
    return H


def transform_point(H, point):
    pt = np.array([[point]], dtype=np.float32)
    transformed = cv2.perspectiveTransform(pt, H)
    return float(transformed[0, 0, 0]), float(transformed[0, 0, 1])


manual_H = None
if CONFIG["homography_mode"] == "manual":
    manual_src = np.array(CONFIG["manual_homography_src"], dtype=np.float32)
    manual_H, _ = cv2.findHomography(manual_src, CANONICAL_DST)


In [7]:
# ============================================================
# Tactical stats — computed from real-world (or fallback-approximate) pitch positions.
# ============================================================
def get_pitch_positions(detections, H, image_size):
    img_w, img_h = image_size
    for d in detections:
        gx, gy = d["ground_point_px"]
        if H is not None:
            px, py = transform_point(H, (gx, gy))
        else:
            nx, ny = gx / img_w, gy / img_h
            px, py = nx * PITCH_W, (1 - ny) * PITCH_H
        d["pitch_x"], d["pitch_y"] = float(np.clip(px, 0, PITCH_W)), float(np.clip(py, 0, PITCH_H))
    return detections


def team_shape_stats(team_positions):
    if len(team_positions) == 0:
        return None
    pts = np.array(team_positions)
    centroid = pts.mean(axis=0)
    xs, ys = pts[:, 0], pts[:, 1]
    width = float(ys.max() - ys.min())   # spread across pitch SHORT axis
    depth = float(xs.max() - xs.min())   # spread across pitch LONG axis
    hull_area = None
    if len(pts) >= 3:
        hull = cv2.convexHull(pts.astype(np.float32))
        hull_area = float(cv2.contourArea(hull))
    return {
        "centroid": [float(centroid[0]), float(centroid[1])],
        "width_m": width,
        "depth_m": depth,
        "aspect_ratio": (width / depth) if depth > 0 else None,
        "convex_hull_area_m2": hull_area,
        "bounding_box": [float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())],
        "num_players": len(pts),
    }


def spacing_stats(team_positions):
    if len(team_positions) < 2:
        return {"avg_nearest_teammate_dist_m": None, "avg_dist_to_centroid_m": None}
    pts = np.array(team_positions)
    centroid = pts.mean(axis=0)
    dists_to_centroid = np.linalg.norm(pts - centroid, axis=1)

    nearest_dists = []
    for i in range(len(pts)):
        others = np.delete(pts, i, axis=0)
        nearest_dists.append(np.min(np.linalg.norm(others - pts[i], axis=1)))

    return {
        "avg_nearest_teammate_dist_m": float(np.mean(nearest_dists)),
        "avg_dist_to_centroid_m": float(np.mean(dists_to_centroid)),
    }


def defensive_line_height(team_outfield_positions):
    # Deepest (smallest x, arbitrary pitch-left convention) outfield player's x position.
    # This is an ABSOLUTE pitch coordinate, not "own goal side" — direction isn't knowable
    # from a single image without extra info about which way this team is attacking.
    if not team_outfield_positions:
        return None
    return float(min(p[0] for p in team_outfield_positions))


def zone_counts(positions, n_zones=3):
    # Absolute pitch thirds along the length (x) axis — NOT "attacking/middle/defensive third",
    # since attacking direction isn't known per image.
    counts = [0] * n_zones
    zone_width = PITCH_W / n_zones
    for x, y in positions:
        idx = min(int(x // zone_width), n_zones - 1)
        counts[idx] += 1
    return counts


def ball_relative_stats(detections, ball_radius_m):
    ball = next((d for d in detections if d["class"] == "ball"), None)
    if ball is None:
        return {"ball_found": False}

    bx, by = ball["pitch_x"], ball["pitch_y"]
    people = [d for d in detections if d["class"] in ("player", "goalkeeper")]
    if not people:
        return {"ball_found": True, "ball_position": [bx, by], "nearest_player_team": None,
                "players_within_radius": {"red": 0, "blue": 0, "unknown": 0}}

    dists = [((p["pitch_x"] - bx) ** 2 + (p["pitch_y"] - by) ** 2) ** 0.5 for p in people]
    nearest_idx = int(np.argmin(dists))

    within_radius = {"red": 0, "blue": 0, "unknown": 0}
    for p, d in zip(people, dists):
        if d <= ball_radius_m:
            within_radius[p.get("team", "unknown")] = within_radius.get(p.get("team", "unknown"), 0) + 1

    return {
        "ball_found": True,
        "ball_position": [bx, by],
        "nearest_player_team": people[nearest_idx].get("team", "unknown"),
        "nearest_player_distance_m": float(dists[nearest_idx]),
        "players_within_radius": within_radius,
        "distance_to_goal_x0_m": float(bx),
        "distance_to_goal_xW_m": float(PITCH_W - bx),
    }


def compute_tactical_stats(detections):
    stats = {}
    for team in ("red", "blue"):
        team_positions = [(d["pitch_x"], d["pitch_y"]) for d in detections
                          if d["class"] in ("player", "goalkeeper") and d.get("team") == team]
        outfield_positions = [(d["pitch_x"], d["pitch_y"]) for d in detections
                              if d["class"] == "player" and d.get("team") == team]

        stats[team] = {
            "shape": team_shape_stats(team_positions),
            "spacing": spacing_stats(team_positions),
            "defensive_line_x": defensive_line_height(outfield_positions),
            "zone_counts": zone_counts(team_positions) if team_positions else None,
        }

    red_centroid = stats["red"]["shape"]["centroid"] if stats["red"]["shape"] else None
    blue_centroid = stats["blue"]["shape"]["centroid"] if stats["blue"]["shape"] else None
    inter_team_dist = None
    if red_centroid and blue_centroid:
        inter_team_dist = float(np.linalg.norm(np.array(red_centroid) - np.array(blue_centroid)))
    stats["inter_team_centroid_distance_m"] = inter_team_dist

    stats["ball"] = ball_relative_stats(detections, CONFIG["ball_radius_m"])
    return stats


In [8]:
# ============================================================
# Scoreboard / clock OCR — reads a fixed pixel region, IF configured. Skipped (not guessed)
# when the corresponding ROI is None.
# ============================================================
def ocr_read_region(image_pil, roi):
    if roi is None or not OCR_AVAILABLE:
        return None
    x1, y1, x2, y2 = roi
    crop = image_pil.crop((x1, y1, x2, y2))
    try:
        text = pytesseract.image_to_string(
            crop, config="--psm 7 -c tessedit_char_whitelist=0123456789:-"
        )
        return text.strip()
    except Exception as e:
        print(f"OCR failed: {e}")
        return None


def read_scoreboard(image_pil):
    return {
        "score_text": ocr_read_region(image_pil, CONFIG["score_roi"]),
        "clock_text": ocr_read_region(image_pil, CONFIG["clock_roi"]),
    }


In [9]:
# ============================================================
# Run: compute + save tactical stats + OCR per qualifying image. Resumable (JSONL, skips
# already-processed image paths), same pattern as the detection metadata pipeline.
# ============================================================
def load_already_done(output_path):
    done = set()
    if os.path.exists(output_path):
        with open(output_path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    done.add(json.loads(line)["image_path"])
                except (json.JSONDecodeError, KeyError):
                    continue
    return done


def process_metadata(name, metadata_path):
    records = load_metadata(metadata_path)
    output_path = os.path.join(CONFIG["stats_output_dir"], f"{name}_tactical_stats.jsonl")
    already_done = load_already_done(output_path)

    print(f"\n--- {name}: {len(records)} records, {len(already_done)} already done ---")

    kept, skipped = 0, 0
    with open(output_path, "a") as out_f:
        for record in records:
            image_path = record["image_path"]
            if image_path in already_done:
                continue

            detections = record["detections"]
            person_count = sum(1 for d in detections if d["class"] in PERSON_CLASSES)
            if person_count < CONFIG["min_person_count"]:
                skipped += 1
                continue
            if not os.path.exists(image_path):
                print(f"WARNING: image not found, skipping: {image_path}")
                continue

            image_pil = Image.open(image_path).convert("RGB")
            image_bgr = cv2.cvtColor(np.array(image_pil), cv2.COLOR_RGB2BGR)

            detections = assign_teams(image_pil, detections, CONFIG["min_players_for_team_split"])
            H = manual_H if CONFIG["homography_mode"] == "manual" else compute_homography(image_bgr)
            detections = get_pitch_positions(detections, H, image_pil.size)

            tactical_stats = compute_tactical_stats(detections)
            scoreboard = read_scoreboard(image_pil)

            out_record = {
                "image_path": image_path,
                "person_count": person_count,
                "homography_used": "manual" if CONFIG["homography_mode"] == "manual"
                                    else ("auto" if H is not None else "fallback_normalization"),
                "tactical_stats": tactical_stats,
                "scoreboard": scoreboard,
            }
            out_f.write(json.dumps(out_record) + "\n")
            out_f.flush()
            kept += 1

    print(f"{name}: processed {kept} new, skipped {skipped} (too few people), "
          f"{len(already_done)} were already done")


for name, path in CONFIG["metadata_paths"].items():
    process_metadata(name, path)



--- train: 2689 records, 0 already done ---
train: processed 2538 new, skipped 151 (too few people), 0 were already done

--- test: 0 records, 0 already done ---
test: processed 0 new, skipped 0 (too few people), 0 were already done


## Notes

- **Zones and defensive line are absolute pitch coordinates**, not "attacking third" or "own
  half" — attacking direction isn't derivable from a single image. If you know which way each
  team attacks for a given video (e.g. from context or a fixed camera setup), you can post-process
  these fields yourself using that knowledge.
- **`homography_used: "fallback_normalization"`** on a record means pitch-corner detection
  failed for that image — treat its tactical stats as less reliable than ones using real
  homography.
- **OCR fields will be `null`/`None`** until you set `score_roi`/`clock_roi` in CONFIG, and even
  then are only as reliable as Tesseract's read of that specific broadcast graphic — spot-check
  a few outputs before trusting them at scale.
- **Goals, shots, possession ratio are intentionally absent** — see this notebook's intro cell
  and the earlier conversation for why those need ordered video or full tracking, neither of
  which fits the discrete-image approach you chose to keep.
